In [ ]:
import requests 

res = requests.post("http://localhost:5000/api/collections",json={
    "name": "evaluation_collection"},
    headers={
        "GuestUserSessionId": "evaluationnotebooksessionid"
    })

In [ ]:
collection_id = res.json()['id']
print(collection_id)

NameError: name 'res' is not defined

In [1]:
import requests
collection_id = 11

c:\Users\Devendra\opensource\ai-research-assistant\backend\venv312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from app.config import Config
client = ChatGoogleGenerativeAI(
                    model="gemini-2.5-flash",
                    api_key=Config.GEMINI_API_KEY,
                    temperature=0.01,
                    streaming=False,
                )

c:\Users\Devendra\opensource\ai-research-assistant\backend\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# upload docs to this collection 
import os
import time
text_files = os.listdir("./rag-dataset/rag-dataset/")

for filename in text_files:
    requests.post(
        url=f"http://localhost:5000/api/documents/upload/{collection_id}",
        files= {"file": open(f"./rag-dataset/rag-dataset/{filename}", "rb")},
        headers={
        "GuestUserSessionId": "evaluationnotebooksessionid"
        }
    )
    time.sleep(5)


In [15]:
# ask question 
import pandas as pd
import json
import time
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
# the answer should be something like "no sufficient context found" 
no_answer_questions = pd.read_csv('no_answer_questions.csv')

for i in range(len(no_answer_questions['question'])):
    question = no_answer_questions['question'].iloc[i]
    res = requests.post(
        url=f"http://localhost:5000/api/chat/query",
        json={
                "question": f"{question}",
                "collection_id": collection_id,
                "model_name": "llama-3.3-70b-versatile",
                "provider": "groq" },
        headers={
        "GuestUserSessionId": "evaluationnotebooksessionid"
        }
    ).json()
    # api call to the LLM judge (gemini 2.5 flash)
    answer = res['answer']
    judge_prompt = f'''
    You are a expert LLM response evaluator. Below is the question asked to an LLM and the LLMs response to that question. Also, there is the Ideal response that the LLM should have given. 

    You have to evaluate the LLMs response against the ideal response and output only a json with below signature
    {{
        'score': '0/1',
        'brief_justification': '...'
    }}

    question: {question}

    answer (LLM): {answer}

    ideal answer: There is not sufficient information in the available context, to answer this question.
    '''
    langchain_messages = []
    langchain_messages.append(HumanMessage(content=judge_prompt))
    judge_response = client.invoke(langchain_messages)
    judge_response = judge_response.content
        # Try to extract JSON from the response
    response = judge_response.strip()

    # Handle case where response is wrapped in markdown code blocks
    if response.startswith("```"):
        lines = response.split("\n")
        json_lines = []
        in_block = False
        for line in lines:
            if line.startswith("```") and not in_block:
                in_block = True
                continue
            elif line.startswith("```") and in_block:
                break
            elif in_block:
                json_lines.append(line)
        response = "\n".join(json_lines)

    evaluation = json.loads(response)
    print(evaluation)
    time.sleep(10)

{'score': '0', 'brief_justification': "The LLM provides additional information ('higher health compared to the regular Bullet Kin') that does not answer the quantitative question 'How much health?' before stating that the exact amount is not specified. The ideal answer is concise and directly states the lack of sufficient information without providing any partial or relative details."}
{'score': '1/1', 'brief_justification': "The LLM's response is identical to the ideal answer, indicating a perfect match."}
{'score': '1', 'brief_justification': "The LLM's response is identical to the ideal response, correctly stating that there is insufficient information to answer the question."}
{'score': '1/1', 'brief_justification': "The LLM's response is identical to the ideal answer, indicating a perfect match."}
{'score': '0', 'brief_justification': 'The LLM provided a detailed answer to the question, but the ideal response states that there was not sufficient information in the available contex

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 39.511536393s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [3]:
# ask question 
import pandas as pd
import json
import time
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
# the answer should be something like "no sufficient context found" 
single_passage_answer_questions = pd.read_csv('single_passage_answer_questions.csv')

for i in range(len(single_passage_answer_questions['question'])):
    question = single_passage_answer_questions['question'].iloc[i]
    ideal_answer = single_passage_answer_questions['answer'].iloc[i]
    res = requests.post(
        url=f"http://localhost:5000/api/chat/query",
        json={
                "question": f"{question}",
                "collection_id": collection_id,
                "model_name": "llama-3.3-70b-versatile",
                "provider": "groq" },
        headers={
        "GuestUserSessionId": "evaluationnotebooksessionid"
        }
    ).json()
    # api call to the LLM judge (gemini 2.5 flash)
    answer = res['answer']
    judge_prompt = f'''
    You are a expert LLM response evaluator. Below is the question asked to an LLM and the LLMs response to that question. Also, there is the correct answer that the LLM response should contain. 

    You have to evaluate the LLMs response against the ideal response and output only a json with below signature
    {{
        'score': '0/1',
        'brief_justification': '...'
    }}

    question: {question}

    answer (LLM): {answer}

    correct answer: {ideal_answer}
    '''
    langchain_messages = []
    langchain_messages.append(HumanMessage(content=judge_prompt))
    judge_response = client.invoke(langchain_messages)
    judge_response = judge_response.content
        # Try to extract JSON from the response
    response = judge_response.strip()

    # Handle case where response is wrapped in markdown code blocks
    if response.startswith("```"):
        lines = response.split("\n")
        json_lines = []
        in_block = False
        for line in lines:
            if line.startswith("```") and not in_block:
                in_block = True
                continue
            elif line.startswith("```") and in_block:
                break
            elif in_block:
                json_lines.append(line)
        response = "\n".join(json_lines)

    evaluation = json.loads(response)
    print(evaluation)
    time.sleep(10)

{'score': '1', 'brief_justification': 'The LLM correctly answers the question about what Keybullet Kin drop. It also provides additional, accurate information about Jammed Keybullet Kin, which is related to the topic and not incorrect.'}
{'score': '1', 'brief_justification': 'The LLM correctly identified the weapon used by the Bandana Bullet Kin as a Machine Pistol, matching the correct answer.'}
{'score': '0', 'brief_justification': "The LLM claimed there was not sufficient information to answer the question, but the 'correct answer' provides a detailed description of two giants, indicating the information was available and the LLM failed to extract it or hallucinated a lack of context."}
{'score': '1', 'brief_justification': "The LLM accurately identifies the two main events that happen on Day 2: emerging in the grotto and encountering 2 Ropers. While it misses some descriptive details and the ropers' motivation present in the correct answer, it correctly answers 'what happens'."}
{'

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 20.677059443s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT